# Improved: retrieve-then-rerank với LLM cục bộ

Baseline cộng thêm Qwen3-8B làm listwise reranker trên các candidate đã truy xuất. Mô hình 8B nạp **4-bit** (~6 GB) nên vừa trên một Colab **T4**. Kết quả chi tiết ở `docs/03_results.md`.

## 1. Kiểm tra runtime

Runtime, Change runtime type, chọn **T4 GPU**.

In [ ]:
!nvidia-smi

## 2. Clone và cài đặt

Colab đã có torch bản CUDA. `pyproject.toml` là nguồn phụ thuộc duy nhất. Cài thêm `.[quant]` để nạp Qwen3-8B ở 4-bit.

In [ ]:
!git clone https://github.com/duongtruongbinh/viettel_ai_race_task2 medextract
%cd medextract
!pip install -e ".[quant]"    # bitsandbytes cho chế độ 4-bit

## 3. Self-check

Không cần GPU, không cần knowledge base. Phải in PASS cho cả bốn mục CONFIG / IMPORTS / SCHEMA / PATHS.

In [ ]:
!python scripts/selfcheck.py

## 4. Knowledge base cho bước linking

Improved kế thừa pipeline baseline nên cũng dùng SapBERT retrieval qua FAISS: cần build cả parquet lẫn index. Danh mục ICD-10 tiếng Việt (TT06) **đã đi kèm repo** tại `data/kb/raw/`, nên bạn chỉ cần tải RxNorm và đặt vào `data/kb/raw/RXNCONSO.RRF` (xem `INSTALL.md`).

In [ ]:
import pathlib
from google.colab import files

pathlib.Path("data/kb/raw").mkdir(parents=True, exist_ok=True)
%cd data/kb/raw
files.upload()          # chỉ cần tải file RxNorm (RXNCONSO.RRF). TT06 .xlsx đã có sẵn trong repo.
%cd /content/medextract
!ls -la data/kb/raw

In [ ]:
!python -m medextract.kb.build_rxnorm          # -> data/kb/processed/rxnorm_terms.parquet
!python -m medextract.kb.build_icd             # -> data/kb/processed/icd_terms.parquet
!python -m medextract.kb.index --device auto   # -> data/kb/processed/{RXNORM,ICD10}.faiss

## 5. Chạy improved và đóng gói bản nộp

Hai bệnh án mẫu đã có sẵn trong `examples/input/`; cell dưới copy chúng vào `data/input/` để chạy. Lần chạy đầu tải Qwen3-8B (vài phút). `configs/improved.yaml` đặt `load_in_4bit: true`. `--zip` ghi `out/improved/submission.zip`, các file JSON nằm phẳng, không có thư mục con.

In [ ]:
!mkdir -p data/input && cp examples/input/*.txt data/input/
!python run.py --config configs/improved.yaml --input data/input \
               --output out/improved --zip
!ls -la out/improved

## 6. Xem một mẫu output

In [ ]:
import json, pathlib

p = pathlib.Path("out/improved/001.json")
data = json.load(open(p, encoding="utf-8"))
print(f"{p.name}: {len(data)} concept(s)\n")
print(json.dumps(data[:3], ensure_ascii=False, indent=2))

## 7. Chấm điểm local

`score.py` là bản đọc lại công thức của Ban Tổ chức để xếp hạng hai lần chạy local, không phải bộ chấm chính thức. Chuẩn bị thư mục nhãn dạng `<thư mục nhãn>/{stem}.json` cùng schema với bản nộp, rồi:

```bash
python score.py --pred out/improved --gold <thư mục nhãn> -v
```